<a href="https://colab.research.google.com/github/chandupradeep18/Python_pandas/blob/main/Weather_Data_ETL_Pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install pandas duckdb requests

In [ ]:
import pandas as pd
import requests
import duckdb
from datetime import datetime
import logging
import time
WEATHER_API_URL = "https://api.open-meteo.com/v1/forecast"
GEO_URL = "https://geocoding-api.open-meteo.com/v1/search"
HOURLY_VARIABLES = "temperature_2m,precipitation,wind_speed_10m,relative_humidity_2m"
cities = ["Visakhapatnam", "Vijayawada", "Guntur", "Nellore", "Kurnool",
    "Tirupati", "Kakinada", "Rajahmundry", "Kadapa", "Anantapur","Eluru", "Ongole", "Vizianagaram", "Srikakulam", "Machilipatnam","Tenali", "Proddatur", "Nandyal", "Adoni", "Hindupur",
    "Bhimavaram", "Tadepalligudem", "Gudivada", "Guntakal", "Dharmavaram","Chittoor", "Madanapalle", "Srikalahasti", "Kavali", "Amalapuram","Tanuku", "Palakollu", "Narasaraopet", "Chilakaluripet", "Markapuram",
    "Tadpatri", "Yemmiganur", "Kadiri", "Rayachoti", "Puttaparthi","Amaravati", "Mangalagiri", "Tadepalli", "Nuzvid", "Jaggayyapeta","Parvathipuram", "Bapatla", "Repalle", "Addanki", "Pithapuram"]
TEST_CITIES = {
    "New York": {"latitude": 40.7128, "longitude": -74.0060,"timezone":"America/New_York"},
    "Los Angeles": {"latitude": 34.0522, "longitude": -118.2437,"timezone":"America/Los_Angeles"},
    "Chicago": {"latitude": 41.8781, "longitude": -87.6298,"timezone":"America/Chicago"},
    "London": {"latitude": 51.5074, "longitude": -0.1278,"timezone":"Europe/London"},
    "Paris": {"latitude": 48.8566, "longitude": 2.3522,"timezone":"Europe/Paris"},
    "Tokyo": {"latitude": 35.6762, "longitude": 139.6503,"timezone":"Asia/Tokyo"},
    "Sydney": {"latitude": -33.8688, "longitude": 151.2093,"timezone":"Australia/Sydney"},
}

In [ ]:
def fetch_city_location(city_names:list)->dict():
  geo_city=dict()
  not_found_city=[]
  for city in city_names:
    params = {"name": city,'country_code':'IN',"count": 1,"language": "en","format": "json"}
    geo_response=requests.get(GEO_URL,params=params,timeout=30)
    geo_response.raise_for_status()
    geo_data=geo_response.json()
    results = geo_data.get("results", [])
    if results:geo_city[city]={"city_name":city,"country_name":geo_data['results'][0]['country'],"latitude":geo_data['results'][0]['latitude'],"longitude":geo_data['results'][0]['longitude'],"timezone":geo_data['results'][0]['timezone']}
    else:not_found_city.append(city)
  if not_found_city:print(f"Total : {len(not_found_city)} Have Invaild City Name or Not City Found : {not_found_city}")
  if geo_city:
    print(f"Total : {len(geo_city)} Cities Geo Location Details Have Been Fetched")
    return geo_city
  return None
city_location=fetch_city_location(cities)

Total : 2 Have Invaild City Name or Not City Found : ['Chilakaluripet', 'Yemmiganur']
Total : 48 Cities Geo Location Details Have Been Fetched


In [ ]:
def extract(CITIES:dict)->list[dict]:
  raw_data =[]
  for name,details in CITIES.items():
    params={"latitude": details["latitude"],"longitude": details["longitude"],"hourly": HOURLY_VARIABLES,"timezone": details["timezone"],}
    response=requests.get(API_URL,params=params,timeout=30)
    response.raise_for_status()
    data = response.json()
    data["city_name"] = name
    raw_data.append(data)
    #print(f"Extracted data for {name}: {len(data['hourly']['time'])} hours")
  print(f"Total {len(raw_data)} Cities Weather Data have Been Fetched")
  return raw_data
if city_location:
  city_weather_date=extract(city_location)
else:
  print("No City Found to Get Weather Data")

Total 48 Cities Weather Data have Been Fetched


In [ ]:
city_weather_date[0]

In [ ]:
def transform(data:list[dict])->pd.DataFrame:
  dataframes =[]
  for city_data in data:
    hourly = city_data["hourly"]
    df = pd.DataFrame({"city": city_data["city_name"],"timestamp": hourly["time"],"temperature_2m": hourly["temperature_2m"],"precipitation": hourly["precipitation"],
                       "wind_speed_10m": hourly["wind_speed_10m"],"relative_humidity_2m": hourly["relative_humidity_2m"],})
    dataframes.append(df)
  combined_df=pd.concat(dataframes,ignore_index=True)
  combined_df['timestamp']=pd.to_datetime(combined_df['timestamp'])
  combined_df["date"]=combined_df['timestamp'].dt.day
  combined_df["hour"]=combined_df['timestamp'].dt.hour
  combined_df['feels_cold']=combined_df['temperature_2m']<10
  combined_df=combined_df[['city','timestamp','date','hour','temperature_2m','precipitation','wind_speed_10m','relative_humidity_2m','feels_cold']]
  print(combined_df.shape)
  return combined_df
if city_weather_date:
  weather_df=transform(city_weather_date)
else:
  print("No City Found to Get Weather Data")

(8064, 9)


In [ ]:
weather_df.sample(5)

,city,timestamp,date,hour,temperature_2m,precipitation,wind_speed_10m,relative_humidity_2m,feels_cold
3345,Hindupur,2026-10-22 09:00:00,22,9,25.7,0.1,10.1,77,False
1346,Kadapa,2026-10-16 02:00:00,16,2,25.5,0.0,2.1,91,False
5846,Tadpatri,2026-10-21 14:00:00,21,14,26.2,0.7,13.3,88,False
3021,Nandyal,2026-10-22 21:00:00,22,21,25.4,0.0,4.0,88,False
6541,Amaravati,2026-10-22 13:00:00,22,13,28.6,0.6,12.6,78,False


In [ ]:
def load(df: pd.DataFrame, db_path: str = "weather.duckdb") -> None:
  con = duckdb.connect(db_path)
  con.execute("DROP TABLE IF EXISTS weather_forecasts")
  con.execute("""
  CREATE TABLE weather_forecasts (
      city STRING,
      timestamp TIMESTAMP,
      date INTEGER,
      hour INTEGER,
      temperature_2m DOUBLE,
      precipitation DOUBLE,
      wind_speed_10m DOUBLE,
      relative_humidity_2m DOUBLE,
      feels_cold BOOLEAN
  )""")
  con.execute("INSERT INTO weather_forecasts SELECT * FROM df")
  row_count = con.execute("SELECT count(*) FROM weather_forecasts").fetchone()[0]
  print(f"Loaded {row_count} rows into DuckDB")
  con.close()
load(weather_df)

Loaded 8064 rows into DuckDB


In [ ]:
def analyze(db_path: str = "weather.duckdb") -> None:
    conn = duckdb.connect(db_path)
    result = conn.execute(
        """
        SELECT
            city,
            ROUND(AVG(temperature_2m), 1) AS avg_temp_c,
            ROUND(SUM(precipitation), 1) AS total_precip_mm,
            ROUND(MAX(wind_speed_10m), 1) AS max_wind_kmh
        FROM weather_forecasts
        GROUP BY city
        ORDER BY avg_temp_c DESC
    """
    ).fetchdf()
    print(result.to_string(index=False))
    print("\n--- Hottest Hour per City (Window Function) ---")
    result = conn.execute(
        """
        SELECT city, timestamp, temperature_2m
        FROM (
            SELECT
                city,
                timestamp,
                temperature_2m,
                RANK() OVER (PARTITION BY city ORDER BY temperature_2m DESC) AS temp_rank
            FROM weather_forecasts
        )
        WHERE temp_rank = 1
    """
    ).fetchdf()
    print(result.to_string(index=False))
        # Query 3: Rainiest day across all cities
    print("\n--- Rainiest Day Across All Cities ---")
    result = conn.execute(
        """
        SELECT
            date,
            ROUND(SUM(precipitation), 2) AS total_precip_mm,
            COUNT(DISTINCT city) AS cities_reporting
        FROM weather_forecasts
        GROUP BY date
        ORDER BY total_precip_mm DESC
        LIMIT 5
    """
    ).fetchdf()
    print(result.to_string(index=False))

    # Summary
    summary = conn.execute(
        """
        SELECT
            COUNT(*) AS total_rows,
            COUNT(DISTINCT city) AS cities,
            MIN(date) AS start_date,
            MAX(date) AS end_date
        FROM weather_forecasts
    """
    ).fetchone()
    print(f"\n--- Pipeline Summary ---")
    print(f"Total rows: {summary[0]}")
    print(f"Cities: {summary[1]}")
    print(f"Date range: {summary[2]} to {summary[3]}")

    conn.close()

In [ ]:
analyze()

          city  avg_temp_c  total_precip_mm  max_wind_kmh
       Kurnool        28.0             15.8          17.3
  Jaggayyapeta        27.9             17.4          16.9
      Kakinada        27.8             29.0          22.4
       Nandyal        27.6             24.4          18.4
    Markapuram        27.6             25.3          18.9
   Rajahmundry        27.3             33.6          18.7
     Amaravati        27.3             29.2          15.0
    Vijayawada        27.2             37.0          16.4
     Proddatur        27.2             41.0          21.6
  Narasaraopet        27.2             33.7          15.0
     Tadepalli        27.2             32.4          15.9
        Ongole        27.1             76.1          15.2
      Tadpatri        27.1             41.8          16.4
   Mangalagiri        27.1             32.4          16.6
 Visakhapatnam        27.1             19.0          17.5
        Kavali        27.1            109.6          21.8
       Addanki